# XP Exercises: Flower Classification using CNN

Week 10, Day 6

**What you will learn**
- Building a CNN for multi class image classification
- Data loading and preprocessing with `image_dataset_from_directory`
- Image visualization techniques
- Model architecture design, compilation, and training
- Evaluating model performance with accuracy and loss plots

**What you will create**: a CNN model that classifies 14 flower species. All parts form one continuous exercise.

## Dataset
Flower classification with 14 classes. Images are organized in class folders, with a provided `train` / `val` split.

The code below downloads the zip if it is not already present, extracts it and locates the dataset root automatically.

In [ ]:
import os, sys, zipfile, shutil, glob, math, json, random, urllib.request
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # hide TensorFlow C++ info logs
from pathlib import Path

DATA_ZIP = Path("./Flower Classification.zip")
EXTRACT_DIR = Path("./data/flower_data")
DATA_URL = ("https://media.githubusercontent.com/media/devtlv/Datasets-GEN-AI-Bootcamp/main/"
            "Week%206/W6D3/Flower%20Classification.zip")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Download the zip if needed (e.g. in a fresh Colab runtime)
if not DATA_ZIP.exists():
    print("Downloading dataset (~215 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_ZIP)

# Extract once
marker = EXTRACT_DIR / ".extracted"
if not marker.exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    marker.write_text("ok")
    print("Extracted:", DATA_ZIP.name, "->", EXTRACT_DIR)
else:
    print("Already extracted. Skipping.")

# Find the dataset root: the folder that contains train/ and val/
def list_dirs(p):
    return [d for d in Path(p).iterdir() if d.is_dir()]

candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    names = [d.name.lower() for d in list_dirs(root)]
    if "train" in names and "val" in names:
        candidates.append(Path(root))
candidates = sorted(set(candidates))
print("Candidate dataset roots:", [str(c) for c in candidates][:5])

## Part 1. Data exploration and visualization

> **Image size:** the original images are up to 800×800 pixels. To keep training fast we resize them to **64×64** (a compromise between the suggested 32/48 and enough detail to see petals and flower centres).

The provided `val` folder only has **98 images (7 per class)**: this is too small to compare many experiments reliably. So we:
- keep the provided `val` folder as a **final test set**, never used for tuning;
- split the `train` folder into **85% training / 15% validation** for model selection and early stopping.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (64, 64)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

base = candidates[0]
train_dir = base / "train"
test_dir = base / "val"
print("Train dir:", train_dir, "| Test dir:", test_dir)

In [ ]:
def load(directory, subset=None, image_size=IMG_SIZE, shuffle=True):
    kwargs = dict(validation_split=0.15, subset=subset) if subset else {}
    return tf.keras.utils.image_dataset_from_directory(
        directory, image_size=image_size, batch_size=None, seed=SEED, label_mode="int", shuffle=shuffle, **kwargs)

train_raw = load(train_dir, "training")
val_raw = load(train_dir, "validation")
test_raw = load(test_dir, shuffle=False)

class_names = train_raw.class_names
num_classes = len(class_names)
print("Classes:", num_classes, class_names)

# Cache and prefetch (images are kept unbatched so that we can change the batch size later)
train_raw, val_raw, test_raw = train_raw.cache(), val_raw.cache(), test_raw.cache()

def prepare(ds, batch_size=BATCH_SIZE, shuffle=False):
    if shuffle:
        ds = ds.shuffle(4000, seed=SEED)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = prepare(train_raw, shuffle=True)
val_ds = prepare(val_raw)
test_ds = prepare(test_raw)

In [ ]:
from collections import Counter

def count_images_per_class(root):
    counts = {}
    for cls in class_names:
        folder = Path(root) / cls
        counts[cls] = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".gif"})
    return counts

counts = pd.DataFrame({'train folder': count_images_per_class(train_dir), 'val folder (test set)': count_images_per_class(test_dir)})
display(counts)
print("Total:", counts.sum().to_dict())

counts['train folder'].sort_values().plot(kind='barh', figsize=(8, 5), color='seagreen')
plt.title('Number of training images per class')
plt.xlabel('images')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation dataset at a higher resolution so the flowers are easier to see
viz_raw = tf.keras.utils.image_dataset_from_directory(train_dir, image_size=(128, 128), batch_size=None, seed=SEED, shuffle=True)

def visualize_images(dataset, class_names, per_class=9):
    """Display one 3x3 grid of images for each class, with the class name as the grid title."""
    collected = {i: [] for i in range(len(class_names))}
    for img, label in dataset:
        l = int(label)
        if len(collected[l]) < per_class:
            collected[l].append(img.numpy().astype('uint8'))
        if all(len(v) == per_class for v in collected.values()):
            break
    for l, imgs in collected.items():
        fig, axes = plt.subplots(3, 3, figsize=(4.5, 4.8))
        for ax, im in zip(axes.ravel(), imgs):
            ax.imshow(im)
            ax.axis('off')
        fig.suptitle(class_names[l], fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

visualize_images(viz_raw, class_names)

PART1_PLACEHOLDER

**Learning point**
Vision models learn features from texture, color, and shape. Dataset bias and imbalance can dominate results without careful preprocessing and evaluation.

## Part 2. Model architecture design

We start from the provided baseline, then design an improved variant, and train both in the same conditions (Adam, up to 15 epochs, early stopping on the validation loss).

In [ ]:
from tensorflow.keras import models

def build_baseline(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),  # safety if datasets were not normalized
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline(num_classes)
baseline.summary()

In [ ]:
def conv_block(x, filters, kernel_size=3):
    x = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return layers.MaxPooling2D()(x)

def build_variant(num_classes, dropout=0.4, optimizer="adam"):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = layers.Rescaling(1./255)(inputs)
    x = conv_block(x, 32, kernel_size=5)   # 64 -> 32, larger 5x5 kernels in the first block
    x = conv_block(x, 64)                  # 32 -> 16
    x = conv_block(x, 128)                 # 16 -> 8
    x = conv_block(x, 256)                 # 8 -> 4
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs, name="flower_cnn_variant")
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model_variant = build_variant(num_classes)
model_variant.summary()

In [ ]:
import time

def fit_model(model, train_ds, val_ds, epochs=5, callbacks=None):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    return history, dt

def plot_curves(history, title="Training"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history.get("accuracy", []), label="acc")
    axes[0].plot(history.history.get("val_accuracy", []), label="val_acc")
    axes[0].set_title(f"{title} - accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history.history.get("loss", []), label="loss")
    axes[1].plot(history.history.get("val_loss", []), label="val_loss")
    axes[1].set_title(f"{title} - loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()

def early_stop(patience=4):
    return tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

arch_results = {}
for name, builder in [("baseline", build_baseline), ("variant", build_variant)]:
    tf.keras.utils.set_random_seed(SEED)
    m = builder(num_classes)
    hist, dur = fit_model(m, train_ds, val_ds, epochs=15, callbacks=[early_stop()])
    plot_curves(hist, title=name)
    arch_results[name] = {"params": m.count_params(), "epochs run": len(hist.history["loss"]),
                          "best val_acc": max(hist.history["val_accuracy"]),
                          "train acc at best epoch": hist.history["accuracy"][int(np.argmin(hist.history["val_loss"]))],
                          "time (s)": round(dur)}
    if name == "variant":
        model_variant = m

display(pd.DataFrame(arch_results).T)

PART2_PLACEHOLDER

## Part 3. Hyperparameter tuning

We tune the improved architecture: optimizer (Adam, RMSprop, SGD with momentum), learning rate and batch size. Every run uses early stopping (patience 3, best weights restored) and at most 12 epochs. The last run adds a learning-rate schedule (`ReduceLROnPlateau`).

In [ ]:
def make_optimizer(name, lr):
    if name == "adam":
        return tf.keras.optimizers.Adam(lr)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(lr)
    return tf.keras.optimizers.SGD(lr, momentum=0.9, nesterov=True)

opts = [
    ("adam", 1e-3, 32, False),
    ("adam", 5e-4, 32, False),
    ("adam", 1e-3, 64, False),
    ("rmsprop", 1e-3, 32, False),
    ("sgd", 1e-2, 64, False),
    ("adam", 1e-3, 32, True),   # + ReduceLROnPlateau schedule
]
results = []
histories = {}
for opt_name, lr, batch, schedule in opts:
    tf.keras.utils.set_random_seed(SEED)
    model = build_variant(num_classes, optimizer=make_optimizer(opt_name, lr))
    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
    if schedule:
        cb.append(tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5))
    hist, dur = fit_model(model, prepare(train_raw, batch, shuffle=True), prepare(val_raw, batch), epochs=12, callbacks=cb)
    label = f"{opt_name} lr={lr} bs={batch}" + (" + LR schedule" if schedule else "")
    histories[label] = hist.history
    results.append({"opt": opt_name, "lr": lr, "batch": batch, "lr schedule": schedule,
                    "best_val_acc": float(max(hist.history["val_accuracy"])),
                    "best_val_loss": float(min(hist.history["val_loss"])),
                    "epochs": len(hist.history["loss"]), "time_s": round(dur, 1)})

results_df = pd.DataFrame(results).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, h in histories.items():
    ax.plot(range(1, len(h["val_accuracy"]) + 1), h["val_accuracy"], "o-", markersize=3, label=label)
ax.set_title("Validation accuracy per epoch for each hyperparameter setting")
ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

PART3_PLACEHOLDER

## Part 4. Data augmentation

We use `ImageDataGenerator` as requested. To keep training fast, the training images (already resized to 64×64) are loaded once into memory and augmented on the fly with `datagen.flow(...)`; the validation set is the same as in the previous parts, so the results are directly comparable.

> Note: the model already contains a `Rescaling(1./255)` layer, so the generator must **not** rescale again (otherwise the pixels would be divided by 255 twice).

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

X_train = np.stack([x.numpy() for x, _ in train_raw]).astype("float32")
y_train = np.array([int(y) for _, y in train_raw])
print("In-memory training set:", X_train.shape, y_train.shape)

augmentations = {
    "no augmentation": dict(),
    "flip only": dict(horizontal_flip=True, vertical_flip=True),
    "rotation + flip": dict(rotation_range=30, horizontal_flip=True, vertical_flip=True),
    "zoom + shift + shear": dict(zoom_range=0.2, width_shift_range=0.1, height_shift_range=0.1, shear_range=0.1),
    "all combined": dict(rotation_range=30, horizontal_flip=True, vertical_flip=True, zoom_range=0.2,
                         width_shift_range=0.1, height_shift_range=0.1, shear_range=0.1, fill_mode="reflect"),
}

# Preview of the full augmentation on one image
datagen_preview = ImageDataGenerator(**augmentations["all combined"])
sample = X_train[:1]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(sample[0].astype("uint8")); axes[0].set_title("original"); axes[0].axis("off")
for ax, batch in zip(axes[1:], datagen_preview.flow(sample, batch_size=1, seed=SEED)):
    ax.imshow(batch[0].astype("uint8")); ax.axis("off")
fig.suptitle(f"Augmented versions of one '{class_names[y_train[0]]}' image")
plt.tight_layout(); plt.show()

In [ ]:
best = results_df.iloc[0]
print(f"Using the best hyperparameters from Part 3: {best['opt']} lr={best['lr']} batch={best['batch']}")

aug_results = {}
aug_histories = {}
for name, params in augmentations.items():
    tf.keras.utils.set_random_seed(SEED)
    datagen = ImageDataGenerator(**params)
    flow_train = datagen.flow(X_train, y_train, batch_size=int(best["batch"]), shuffle=True, seed=SEED)
    model_aug = build_variant(num_classes, optimizer=make_optimizer(best["opt"], float(best["lr"])))
    hist_aug, dur = fit_model(model_aug, flow_train, prepare(val_raw, int(best["batch"])), epochs=20,
                              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])
    aug_histories[name] = hist_aug.history
    i_best = int(np.argmin(hist_aug.history["val_loss"]))
    aug_results[name] = {"best val_acc": max(hist_aug.history["val_accuracy"]),
                         "train acc at best epoch": hist_aug.history["accuracy"][i_best],
                         "val acc at best epoch": hist_aug.history["val_accuracy"][i_best],
                         "epochs": len(hist_aug.history["loss"]), "time (s)": round(dur)}

aug_df = pd.DataFrame(aug_results).T
aug_df["overfitting gap"] = aug_df["train acc at best epoch"] - aug_df["val acc at best epoch"]
display(aug_df.sort_values("best val_acc", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in aug_histories.items():
    axes[0].plot(h["val_accuracy"], label=name)
    axes[1].plot(h["val_loss"], label=name)
axes[0].set_title("Validation accuracy"); axes[1].set_title("Validation loss")
for ax in axes: ax.set_xlabel("epoch")
axes[0].legend(fontsize=9)
plt.tight_layout(); plt.show()

PART4_PLACEHOLDER

**Learning point**
Augmentation encodes invariances like rotation and translation. It increases effective sample diversity which often reduces overfitting.

## Part 5. Performance evaluation and analysis

**Final model:** improved architecture + best hyperparameters + best augmentation, trained longer (up to 40 epochs) with early stopping and a `ReduceLROnPlateau` schedule.

In [ ]:
best_aug_name = aug_df["best val_acc"].idxmax()
print("Best augmentation:", best_aug_name)

tf.keras.utils.set_random_seed(SEED)
datagen = ImageDataGenerator(**augmentations[best_aug_name])
flow_train = datagen.flow(X_train, y_train, batch_size=int(best["batch"]), shuffle=True, seed=SEED)
best_model = build_variant(num_classes, optimizer=make_optimizer(best["opt"], float(best["lr"])))
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]
final_hist, final_dur = fit_model(best_model, flow_train, prepare(val_raw, int(best["batch"])), epochs=40, callbacks=callbacks)
print(f"Training time: {final_dur / 60:.1f} min, epochs run: {len(final_hist.history['loss'])}")
plot_curves(final_hist, title="Final model")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def collect_preds(model, ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pr = model.predict(xb, verbose=0)
        y_prob.append(pr)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_pred = y_prob.argmax(axis=1)
    else:
        y_pred = (y_prob.ravel() >= 0.5).astype(int)
    return y_true, y_pred, y_prob

def plot_confusion(cm, labels, title="Confusion matrix"):
    plt.figure(figsize=(9, 8))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=90)
    plt.yticks(ticks, labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] > 0:
                plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                         color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(fraction=0.046)
    plt.tight_layout()
    plt.show()

In [ ]:
for name, ds in [("Validation split (15% of train)", val_ds), ("Test set (provided val folder)", test_ds)]:
    loss, acc = best_model.evaluate(ds, verbose=0)
    print(f"{name:<35} loss = {loss:.4f} | accuracy = {acc:.4f}")

# Detailed metrics on the validation split (~2,000 images, more reliable than the 98 test images)
y_true_v, y_pred_v, y_prob_v = collect_preds(best_model, val_ds)
print("\nClassification report - validation split")
print(classification_report(y_true_v, y_pred_v, target_names=class_names, digits=3))
plot_confusion(confusion_matrix(y_true_v, y_pred_v), class_names, "Confusion matrix - validation split")

In [ ]:
y_true_t, y_pred_t, y_prob_t = collect_preds(best_model, test_ds)
print("Classification report - test set (98 images, 7 per class)")
print(classification_report(y_true_t, y_pred_t, target_names=class_names, digits=3, zero_division=0))
plot_confusion(confusion_matrix(y_true_t, y_pred_t), class_names, "Confusion matrix - test set")

In [ ]:
# Per-class difficulty and most frequent confusions (validation split)
cm_v = confusion_matrix(y_true_v, y_pred_v)
per_class = pd.DataFrame({"images": cm_v.sum(axis=1), "recall": np.diag(cm_v) / cm_v.sum(axis=1)}, index=class_names)
per_class = per_class.sort_values("recall")
display(per_class.style.format({"recall": "{:.1%}"}))

off = cm_v.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], class_names[i], class_names[j]) for i in range(num_classes) for j in range(num_classes) if off[i, j] > 0), reverse=True)
print("Most frequent confusions (true -> predicted):")
for n, a, b in pairs[:8]:
    print(f"  {a} -> {b}: {n}")

In [ ]:
# Predictions on the test images: correct (green) and misclassified (red)
test_imgs = np.stack([x.numpy() for x, _ in test_raw]).astype("uint8")
order = list(np.where(y_pred_t != y_true_t)[0][:12]) + list(np.where(y_pred_t == y_true_t)[0][:12])
fig, axes = plt.subplots(4, 6, figsize=(15, 11))
for ax, k in zip(axes.ravel(), order):
    ax.imshow(test_imgs[k])
    ok = y_pred_t[k] == y_true_t[k]
    ax.set_title(f"true: {class_names[y_true_t[k]]}\npred: {class_names[y_pred_t[k]]} ({y_prob_t[k].max():.0%})",
                 color="green" if ok else "red", fontsize=9)
    ax.axis("off")
fig.suptitle("Test images - first rows: misclassified, last rows: correctly classified", fontsize=13)
plt.tight_layout()
plt.show()

PART5_PLACEHOLDER

## Part 6. Model saving and deployment (optional)

In [ ]:
os.makedirs("./data", exist_ok=True)
best_model.save("./data/flower_cnn.keras")  # native Keras format (recommended)
best_model.save("./data/flower_cnn.h5")     # legacy HDF5 file
print("Saved to ./data/:", [f for f in os.listdir("./data") if f.startswith("flower_cnn")])

# Check that the saved model gives the same predictions
reloaded = tf.keras.models.load_model("./data/flower_cnn.keras")
_, acc_reloaded = reloaded.evaluate(test_ds, verbose=0)
print(f"Reloaded model - test accuracy: {acc_reloaded:.4f}")

def predict_flower(path, model=reloaded):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    x = tf.keras.utils.img_to_array(img)[None, ...]
    probs = model.predict(x, verbose=0)[0]
    top = probs.argsort()[::-1][:3]
    return [(class_names[i], float(probs[i])) for i in top]

example = next((test_dir / "rose").glob("*.jpg"))
print("Top-3 prediction for", example.name, ":", predict_flower(example))

**Deployment options:**

- **Web API with Flask:** load the saved `.keras` model once at start-up and expose an endpoint that receives an image and returns the predicted species:

```python
from flask import Flask, request, jsonify
import tensorflow as tf, numpy as np, io
from PIL import Image

app = Flask(__name__)
model = tf.keras.models.load_model("flower_cnn.keras")
CLASS_NAMES = [...]  # same order as during training

@app.route("/predict", methods=["POST"])
def predict():
    img = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB").resize((64, 64))
    probs = model.predict(np.array(img, dtype="float32")[None, ...], verbose=0)[0]
    return jsonify({"species": CLASS_NAMES[int(probs.argmax())], "confidence": float(probs.max())})
```

- **Cloud:** package this API in a Docker container and deploy it on Google Cloud Run, AWS (SageMaker endpoint or Lambda) or Azure, or serve the SavedModel with TensorFlow Serving / Vertex AI.
- **Mobile:** convert the model to **TensorFlow Lite** (`tf.lite.TFLiteConverter.from_keras_model(best_model)`) to run predictions directly on a phone, even offline, e.g. in a "which flower is this?" camera app.